In [1]:
import os
import pickle
import re
import html
import pandas as pd
import numpy as np
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

## Phase 1: Data Recovery and Natural-Language Text Preprocessing
In the original Data Mining pipeline, textual descriptions were dropped to focus on numerical analysis.
Here, the natural-language description is recovered from the raw BGG details and aligned with the cleaned dataset using `BGGId`.

The same natural-language text is used as the common source for the retrieval pipeline. For BM25, the text is cleaned only at retrieval time: lowercasing, punctuation removal, tokenization, and stop-word removal. The original natural text is preserved in `Description_nat` for inspection and for consistency with the dense-retrieval notebook.

**Caching note**: `df` is always rebuilt fresh from the source CSVs. The tokenized natural-language descriptions are cached to `data/tokens.pkl` because tokenization takes time. The cache is row-aligned with the current `df` and is regenerated after changing the text source.


In [2]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

[nltk_data] Downloading package punkt to /home/tommaso/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
DATA_DIR = '../data'
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')

# 1. Rebuild df from the source CSVs
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)

# 2. Recover the original natural-language description from details.csv
#    BGGId is the stable identifier used to align the two datasets.
orig = pd.read_csv(DETAILS_PATH).drop_duplicates('id')
m = df[['BGGId']].merge(orig[['id', 'description']], left_on='BGGId', right_on='id', how='left')
assert len(m) == len(df), 'merge changed the number of rows'

# Match the cleaning used for the natural-text retrieval pipeline.
def clean_natural_text(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(html.unescape(text))
    return re.sub(r'\s+', ' ', text).strip()

natural_text = m['description'].map(clean_natural_text)
has_natural = natural_text.str.len() > 0
df['Description_nat'] = natural_text.where(has_natural, df['Description']).values

print(f'Dataset ready! Total board games: {len(df)}')
print(f'Games using the original natural-language description: {has_natural.sum()} / {len(df)}')
print(f'Games falling back to DM1 description: {(~has_natural).sum()} / {len(df)}')

# 3. Token cache: it is generated from Description_nat, not the old Description column.
tokens_list = None
if os.path.exists(TOKENS_PATH):
    with open(TOKENS_PATH, 'rb') as f:
        cached_tokens = pickle.load(f)
    if len(cached_tokens) == len(df):
        tokens_list = cached_tokens
        print('Reusing cached tokens from disk...')
    else:
        print(
            f"Cache found at '{TOKENS_PATH}' but row count doesn't match df "
            f"({len(cached_tokens)} vs {len(df)}) - treating as stale, will recompute."
        )

if tokens_list is None:
    print('Tokenizing natural-language descriptions...')
    tokens_list = df['Description_nat'].apply(preprocess_text).tolist()
    with open(TOKENS_PATH, 'wb') as f:
        pickle.dump(tokens_list, f)
    print(f"Tokens computed and cached to '{TOKENS_PATH}'.")

df['tokens'] = tokens_list

print('Sample natural description:')
print(df['Description_nat'].iloc[0][:300])
print('Sample tokenization output:')
print(df['tokens'].iloc[0][:10])

Dataset ready! Total board games: 19728
Games using the original natural-language description: 19256 / 19728
Games falling back to DM1 description: 472 / 19728
Reusing cached tokens from disk...
Sample natural description:
Game description from the publisher: In Assassin's Creed: Arena – based on the Assassin's Creed video game series from Ubisoft – players compete to uncover the best targets for assassination, but before you can go in for the kill, sometimes you have to deal with pesky guards as well as your opponent
Sample tokenization output:
['game', 'description', 'publisher', 'assassins', 'creed', 'arena', '–', 'based', 'assassins', 'creed']


## BM25 Inverted Index Construction
The Okapi BM25 algorithm is used to build an inverted index over the tokens extracted from `Description_nat`. BM25 penalizes overly long documents and gives higher weight to rare terms.

The document text is therefore natural language at the source, while `preprocess_text()` provides the BM25-specific tokenization.

In [4]:
# Extract the list of tokenized descriptions
corpus_tokens = df['tokens'].tolist()

# Initialize the BM25 model
bm25 = BM25Okapi(corpus_tokens)

print(f"BM25 Index successfully built for {len(corpus_tokens)} documents!")

BM25 Index successfully built for 19728 documents!


## Query Processing and Retrieval
The following function processes natural-language user queries with the same `preprocess_text()` function used for `Description_nat`, then retrieves the Top-K most relevant board games.

In [5]:
def search_games_bm25(query, top_k=5):
    """
    Searches the BM25 index for the most relevant games given a natural language query.
    """
    # 1. Preprocess the query using the same pipeline used for Description_nat.
    query_tokens = preprocess_text(query)

    # 2. Get the BM25 scores for all documents in the corpus.
    doc_scores = bm25.get_scores(query_tokens)

    # 3. Use a copy of the dataframe to avoid mutating the shared one during testing.
    df_temp = df.copy()
    df_temp['bm25_score'] = doc_scores

    # 4. Sort results descending by score and get the top K.
    results = df_temp.sort_values(by='bm25_score', ascending=False).head(top_k)

    # Return the game name, score, and the natural-language description.
    return results[['Name', 'bm25_score', 'Description_nat']]

### Test the BM25 search engine
The example below uses a natural-language query and displays the original natural-language description returned by the retrieval system.

In [6]:
user_query = "A cooperative space game involving betrayal"
results = search_games_bm25(user_query, top_k=5)

print(f"Results for: '{user_query}'\n")
for i, row in results.iterrows():
    print(f"{row['Name']} (Score: {row['bm25_score']:.2f})")
    print(f"Snippet: {row['Description_nat'][:150]}...\n")


Results for: 'A cooperative space game involving betrayal'

Betrayal at House on the Hill (Score: 15.68)
Snippet: From the press release: Betrayal at House on the Hill quickly builds suspense and excitement as players explore a haunted mansion of their own design,...

Betrayal at Baldur's Gate (Score: 15.37)
Snippet: Description from the publisher: The shadow of Bhaal has come over Baldur's Gate, summoning monsters and other horrors from the darkness! As you build ...

Room 25 (Score: 15.18)
Snippet: Trapped in a prison in which each room has four doors but apparently no exit, the players must try to find Room 25, the supposed exit to this nightmar...

SSO (Score: 14.93)
Snippet: Semi co-op Sci-fi horror survival cards and counters game. Players attempt to outlast a challenge deck of narrative threats by completing missions and...

Cutthroat Caverns (Score: 14.74)
Snippet: "Without teamwork, you will never survive. Without betrayal, you’ll never win." Cutthroat Caverns is played over 9

Same query as notebook 7, for a direct BM25-vs-FAISS comparison on identical input. BM25 scores reflect lexical relevance after preprocessing the natural-language descriptions.

BM25 is strong on literal terms such as "betrayal" and "cooperative", but it has limited ability to recognize semantic relationships. For example, a query containing "space" may rank a game highly because of overlapping words without fully understanding the intended setting or meaning. Semantic search in the next notebook addresses this complementary limitation using dense embeddings of the same natural-language description source.